<a href="https://colab.research.google.com/github/ZiyuCao421/26-CP5046-SIN-CAM-TR3-ICT-Project-1-Analysis-and-Design/blob/main/V3/notebooks/03_feature_correlation_and_set_definition_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V3 — Stage 3: Feature Correlation Analysis and Candidate Feature-Set Definition

**Purpose.** This notebook does three things and stops there:

1. **Rebuilds** the same complete-case cohort that stage 1 used, from the raw course CSV files.
2. **Computes** feature-to-feature Pearson and Spearman correlations for the features that stage 2
   shortlisted, on that cohort.
3. **Defines** three candidate feature sets, deterministically and without redundancy, as
   *candidates to be evaluated later*.

**Hard boundaries — this notebook never crosses them.**

* **No model is trained.** There is no `fit`, no `predict`, no hyper-parameter search, no scoring
  of a feature set and no champion. The sets are **provisional / not yet evaluated**.
* **No raw data leaves the notebook.** The raw course CSV files are read in the Colab runtime, used
  in memory, and never written, copied, uploaded or committed. No per-row value, no label value,
  no source file name and no local absolute path appears in any artefact.
* **No correlation is fabricated.** Both coefficients are computed from the reconstructed cohort.

> **Results validity.** Every file, figure and table this notebook produces is
> **only valid after a real Colab run on the real course data**. The notebook is committed
> **without stored outputs** and contains **no result value of any kind**.

## Plan A / Plan B — how the inputs are obtained

This notebook is written to work with **both** situations, and it decides at run time which one it
is in.

| Plan | Situation | Behaviour |
| --- | --- | --- |
| **A** | The stage-1 aggregates, the stage-2 aggregates and the raw course CSV files are all readable in the runtime | Everything is read, the input contract is checked against the real artefacts, and the cohort is reconstructed and cross-checked against the stage-1 manifest |
| **B** | One or more inputs are missing | The run **stops with an explicit error naming every missing file**. Nothing is guessed, reconstructed or hard-coded, and no partial conclusion is produced |

The same notebook handles both: the validation cell reports which files were found and which were
missing, and the reconstruction cell refuses to continue on an inconsistent cohort.

### Files this notebook needs

| File | Produced by | Purpose |
| --- | --- | --- |
| `feature_inventory.csv` | stage 1 | The candidate pool: which columns are candidates |
| `run_metadata.json` | stage 1 | Cohort size, class count, split fingerprint, configuration of record |
| `single_feature_metrics.csv` | stage 1 | Per-feature metrics and ranks, and the model roster |
| `selected_features.csv` | stage 2 | The shortlisted candidates and their ranks |
| `stage2_candidate_manifest.json` | stage 2 | Stage-2 status and the split fingerprint it used |
| the raw course CSV files | course data | The values the correlations are computed from (read in the Colab runtime only) |

## Outputs

| File | Content |
| --- | --- |
| `pearson_correlation.csv` | Pearson matrix over the shortlisted features |
| `spearman_correlation.csv` | Spearman matrix over the shortlisted features |
| `correlation_pairs.csv` | Upper triangle: one row per feature pair with both coefficients and the redundancy flags |
| `correlation_summary.json` | Configuration, input hashes, validation report, counts, threshold, limitations |
| `feature_set_definitions.json` | The three candidate sets, their members, their rejected features and the reasons |
| `feature_set_membership.csv` | One row per (set × feature), including the rejected ones and why |
| `correlation_reconstruction_report.md` | The run report: reconstruction, validation, correlation results, the sets, the limitations |
| `figures/pearson_heatmap.png`, `figures/spearman_heatmap.png` | The two correlation heatmaps |
| `figures/redundant_pairs.png` | The pairs that reach the redundancy threshold |

## How the three sets are built

All three sets use **the same deterministic greedy strategy**; only the ordering rule differs.

| Set | Ordering rule |
| --- | --- |
| `FS1_consensus_compact` | stage-2 mean rank of the two real models (RBF SVC and Random Forest) |
| `FS2_svc_diverse` | stage-2 rank under the RBF SVC |
| `FS3_random_forest_diverse` | stage-2 rank under the Random Forest |

The greedy walk takes the features in that order and **accepts** a feature only when its
`|Pearson|` **and** its `|Spearman|` against **every** feature already in the set are **below**
`CORRELATION_THRESHOLD`. It stops at `MAX_FEATURES_PER_SET`. Ties in the ordering are broken by
**the larger of the two models' macro F1 (descending)**, then by **feature name (ascending)**, so
the result never depends on the row order of the input file.

Every rejected feature is recorded with its reason. If a set cannot be built from at least two
features, it is written as **`not_sufficient_nonredundant_candidates`** — never silently
replaced and never presented as a success.

## The `potential_recording_or_session_risk` flag

Features whose name matches an absolute position or time statistic (`avg_x`, `min_y`, `max_t` and
friends) are the aggregate counterparts of the raw coordinates and timestamps that stage 1
excluded. They may encode **where on the screen** and **when in the recording** a segment
occurred, i.e. the recording rather than the person. Such features are **kept** as provisional
candidates, but every artefact in which they appear carries the flag
`potential_recording_or_session_risk`, and this notebook never describes them as robust
identity-behaviour features.


## How to run this notebook in Google Colab

### Before you start

Stage 1 and stage 2 must have been run, and their outputs must exist. Put the five aggregate files
where this notebook can read them:

```text
<STAGE1_OUTPUT_DIR>/     feature_inventory.csv
                         run_metadata.json
                         single_feature_metrics.csv
<STAGE2_OUTPUT_DIR>/     selected_features.csv
                         stage2_candidate_manifest.json
<RAW_DATA_DIR>/          the raw course CSV files (one per participant)
```

The raw folder is read **in the Colab runtime only**. Do not copy it into the repository, do not
upload it anywhere, and do not commit it.

### Steps

1. **Open the notebook in Colab** and set *Runtime → Run all*.
2. **Mount Drive** if the data lives there:

   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```

3. **Edit the single configuration cell** (headed *SINGLE EDITABLE CONFIGURATION BLOCK*). Set
   `RAW_DATA_DIR`, `STAGE1_OUTPUT_DIR`, `STAGE2_OUTPUT_DIR` and `OUTPUT_DIR`. The defaults are
   Colab **template** paths; no real local or Drive location is hard-coded anywhere in this
   notebook. Everything else has a working default.
4. **Run all cells.** The order is: input contract → cohort reconstruction → correlation →
   feature sets → artefacts.
5. **Read the outputs** from `OUTPUT_DIR`: the printed validation report, the reconstruction
   report and the three candidate sets.

### What this notebook will not do

* It will not continue with a missing or inconsistent input. If a file is absent, or the stage-1
  and stage-2 split fingerprints disagree, or the reconstructed cohort does not match the stage-1
  manifest, the run stops with an explicit error and writes no conclusion.
* It will not train, score or champion anything. The feature sets it writes are labelled
  `provisional / not yet evaluated`.
* It will not write raw data, per-row values, label values, source file names or absolute paths.


In [13]:
# =============================================================================
#  ENVIRONMENT
#  Nothing here is editable. It only reports the runtime the results came from.
# =============================================================================
import hashlib
import json
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.use("Agg")  # figures are saved to disk, not displayed inline

print("python      :", sys.version.split()[0])
print("platform    :", platform.platform())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print()
print("No machine-learning library is imported by this notebook: no model is trained here.")


python      : 3.13.18
platform    : Linux-6.6.159+-x86_64-with-glibc2.39
numpy       : 2.4.6
pandas      : 2.2.3
matplotlib  : 3.10.0

No machine-learning library is imported by this notebook: no model is trained here.


In [14]:
# =============================================================================
#  SINGLE EDITABLE CONFIGURATION BLOCK
#  ---------------------------------------------------------------------------
#  Edit ONLY this cell. Every other cell reads these names and nothing else.
#  The paths below are Colab TEMPLATE paths, not a real local or Drive
#  location: point them at the folders that hold the stage-1 outputs, the
#  stage-2 outputs and the raw course CSV files.
# =============================================================================

# --- Raw course data (read in the Colab runtime only) -----------------------
RAW_DATA_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics"
FILE_PATTERN = "*.csv"
LABEL_COLUMN = "label"

# --- Stage outputs (aggregate files from the earlier stages) -----------------
STAGE1_OUTPUT_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics/V3"
STAGE2_OUTPUT_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics/V3"

# --- Where this notebook writes ---------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/CP5046_Mouse_Dynamics/V3/stage3_output"

# --- Protocol parameters (must match stage 1) -------------------------------
MIN_CLASS_COUNT = 5             # classes with fewer rows than this are dropped
RANDOM_STATE = 42
TEST_SIZE = 0.20

# --- Correlation and set construction ---------------------------------------
CORRELATION_THRESHOLD = 0.85    # a pair at or above this |r| is treated as redundant
MAX_FEATURES_PER_SET = 6        # upper bound on the size of each candidate set

# --- Figures ----------------------------------------------------------------
SAVE_FIGURES = True
FIGURE_DPI = 150

## Configuration reference

Every knob lives in the single editable cell above. The paths are Colab **template** paths on
purpose: no real local or Drive location is hard-coded anywhere in this notebook.

| Name | Default | Meaning |
| --- | --- | --- |
| `RAW_DATA_DIR` | Colab template | Folder holding the raw course CSV files. Read in the runtime only; never written, copied or committed. |
| `FILE_PATTERN` | `*.csv` | Glob that selects the per-participant files |
| `LABEL_COLUMN` | `label` | The label column as it appears **inside** the CSV files. The label is never derived from a file name. |
| `STAGE1_OUTPUT_DIR` | Colab template | Folder with `feature_inventory.csv`, `run_metadata.json`, `single_feature_metrics.csv` |
| `STAGE2_OUTPUT_DIR` | Colab template | Folder with `selected_features.csv`, `stage2_candidate_manifest.json` |
| `OUTPUT_DIR` | Colab template | Where this notebook writes its artefacts |
| `MIN_CLASS_COUNT` | `5` | Classes with fewer rows than this are dropped when the cohort is rebuilt. Must match stage 1. |
| `RANDOM_STATE` | `42` | Seed of record, compared against the stage-1 configuration |
| `TEST_SIZE` | `0.20` | Test fraction of record, compared against the stage-1 configuration |
| `CORRELATION_THRESHOLD` | `0.85` | A pair whose absolute Pearson **or** absolute Spearman reaches this value is treated as redundant |
| `MAX_FEATURES_PER_SET` | `6` | Upper bound on the size of each candidate set |
| `SAVE_FIGURES` | `True` | Whether the heatmaps are written |
| `FIGURE_DPI` | `150` | Resolution of the saved PNGs |

### The correlation threshold

For every pair of shortlisted features the notebook computes both coefficients on the
reconstructed cohort and reports them **separately**. A pair is flagged as redundant when

```text
abs(Pearson)  >= CORRELATION_THRESHOLD   or   abs(Spearman) >= CORRELATION_THRESHOLD
```

The rule is *at or above*, so a pair exactly at the threshold counts as redundant. Both
coefficients are used because they fail differently: Pearson detects linear redundancy, Spearman
detects monotone but non-linear redundancy (for example a feature and a rescaled copy of it).
A feature that is constant inside the cohort has no defined correlation; it is reported as
`constant_in_cohort` and excluded from the candidate sets.

### The set construction logic

For each of the three sets the features are visited in a fixed order and accepted greedily:

1. take the next feature in the set's ordering rule;
2. accept it **only if** its `|Pearson|` and its `|Spearman|` against **every** feature already in
   the set are both **below** `CORRELATION_THRESHOLD`;
3. stop when the set holds `MAX_FEATURES_PER_SET` features, or when no further feature qualifies.

The ordering rules differ per set; the acceptance rule, the threshold and the tie-break are
identical for all three, so the sets are comparable with one another.

**Tie-break.** Features with equal rank are ordered by **the larger of the two models' macro F1
(descending)**, then by **feature name (ascending)**. The result therefore never depends on the
row order of the input file, and two runs on the same inputs produce the same sets.

**Rejections are recorded.** Every feature that is not in a set appears in
`feature_set_membership.csv` with its reason — `redundant_with:<feature>` (and which coefficient
triggered it), `set_full`, or `constant_in_cohort`. No feature is ever silently dropped, and no
feature is ever silently substituted.

### Plan A and Plan B

The notebook does not need to be told which plan it is in. The validation cell looks for every
required input and reports:

* **Plan A** — all inputs present and consistent: the run proceeds and the cohort reconstruction
  is cross-checked against the stage-1 manifest.
* **Plan B** — an input is missing or inconsistent: the run stops, naming exactly what is missing
  or what disagrees, and writes no conclusion.


In [15]:
# =============================================================================
#  HELPERS  (not editable)
# =============================================================================
RESULTS_VALIDITY = "only valid after a real Colab run"

STAGE1_INVENTORY = "feature_inventory.csv"
STAGE1_METADATA = "run_metadata.json"
STAGE1_METRICS = "single_feature_metrics.csv"
STAGE2_SELECTED = "selected_features.csv"
STAGE2_MANIFEST = "stage2_candidate_manifest.json"

REQUIRED_STAGE1_FILES = [STAGE1_INVENTORY, STAGE1_METADATA, STAGE1_METRICS]
REQUIRED_STAGE2_FILES = [STAGE2_SELECTED, STAGE2_MANIFEST]

REQUIRED_COLUMNS = {
    STAGE1_INVENTORY: ["column", "status", "exclusion_reason", "dtype"],
    STAGE1_METRICS: [
        "feature", "model", "macro_f1", "cohort_mode", "n_train_common", "n_test_common",
        "n_classes_common", "split_fingerprint", "results_validity",
    ],
    STAGE2_SELECTED: [
        "feature", "rank_rbf_svc", "rank_random_forest", "mean_rank",
        "macro_f1_rbf_svc", "macro_f1_random_forest", "selected", "selection_reason",
        "results_validity", "split_fingerprint",
    ],
}

ALLOWED_VALIDITY = {
    "only valid after a real Colab run",
    "exploratory / not rankable - only valid after a real Colab run",
}

KNOWN_MODELS = {
    "DummyClassifier_most_frequent",
    "RBF_SVC_scaled",
    "RandomForest_balanced",
}

REQUIRED_COHORT_MODE = "complete_case_all_candidates"
REQUIRED_RUN_STATUS = "completed"

# Absolute position / time statistics: the aggregate counterparts of the raw
# coordinates and timestamps that stage 1 excluded.
ABSOLUTE_STATISTIC_PATTERN = re.compile(r"^(avg|min|max)_(x|y|t)$", flags=re.IGNORECASE)

# The three candidate sets and their ordering rules.
FEATURE_SET_RULES = {
    "FS1_consensus_compact": ("mean_rank", "stage-2 mean rank of the RBF SVC and the Random Forest"),
    "FS2_svc_diverse": ("rank_rbf_svc", "stage-2 rank under the RBF SVC"),
    "FS3_random_forest_diverse": ("rank_random_forest", "stage-2 rank under the Random Forest"),
}

SET_STATUS_PROVISIONAL = "provisional / not yet evaluated"
SET_STATUS_INSUFFICIENT = "not_sufficient_nonredundant_candidates"


class InputContractError(RuntimeError):
    """Raised when the inputs do not satisfy the contract of this stage."""


class ReconstructionError(RuntimeError):
    """Raised when the rebuilt cohort does not match the stage-1 manifest."""


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_file(path):
    """SHA-256 of a file's bytes, used to fingerprint the inputs."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def require(condition, message):
    """Raise an explicit contract error when a condition does not hold."""
    if not condition:
        raise InputContractError(message)


def is_absolute_statistic(name):
    """True for absolute position / time statistics such as avg_x or max_t."""
    return bool(ABSOLUTE_STATISTIC_PATTERN.match(str(name)))


def numeric_finite_mask(series):
    """Numeric coercion of a column; returns (values, finite mask).

    Missing values, non-numeric values and +/-inf are all treated as not finite.
    """
    try:
        coerced = pd.to_numeric(series, errors="coerce")
        values = coerced.to_numpy(dtype="float64", na_value=np.nan)
    except (TypeError, ValueError):
        values = np.full(len(series), np.nan, dtype="float64")
    return values, np.isfinite(values)


def spearman_matrix(frame):
    """Spearman correlation without a scipy dependency.

    Spearman's rho is the Pearson correlation of the average ranks, so the
    columns are ranked first and the ordinary Pearson correlation is taken.
    """
    ranks = frame.rank(axis=0, method="average")
    return ranks.corr(method="pearson")


def stamp(frame, validity=RESULTS_VALIDITY):
    """Attach the validity stamp to every row of a table."""
    out = frame.copy()
    out["results_validity"] = validity
    return out


def dump_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, default=str, ensure_ascii=False), encoding="utf-8"
    )
    return path


print("helpers loaded; validity stamp =", repr(RESULTS_VALIDITY))


helpers loaded; validity stamp = 'only valid after a real Colab run'


In [16]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# =============================================================================
#  STEP 1 — INPUT CONTRACT
#  Every input is checked before anything is computed. A failed check stops the
#  run: no correlation, no feature set, no conclusion.
# =============================================================================
validation_results = []


def check(name, condition, detail=""):
    validation_results.append({"check": name, "passed": bool(condition), "detail": detail})
    return bool(condition)


stage1_dir = Path(STAGE1_OUTPUT_DIR).expanduser()
stage2_dir = Path(STAGE2_OUTPUT_DIR).expanduser()
raw_dir = Path(RAW_DATA_DIR).expanduser()

# --- 1. the inputs are there at all (Plan A / Plan B detection) ---------------
missing_directories = [
    name for name, path in (("STAGE1_OUTPUT_DIR", stage1_dir), ("STAGE2_OUTPUT_DIR", stage2_dir),
                            ("RAW_DATA_DIR", raw_dir))
    if not path.is_dir()
]
check(
    "the three input directories exist",
    not missing_directories,
    f"missing: {missing_directories}",
)

missing_files = []
for directory, names, label in (
    (stage1_dir, REQUIRED_STAGE1_FILES, "stage 1"),
    (stage2_dir, REQUIRED_STAGE2_FILES, "stage 2"),
):
    for name in names:
        if not (directory / name).exists():
            missing_files.append(f"{label}: {name}")
check(
    "all five aggregate input files exist",
    not missing_files,
    f"missing: {missing_files}",
)

if missing_directories or missing_files:
    print("INPUT CONTRACT REPORT")
    print("-" * 78)
    for entry in validation_results:
        mark = "PASS" if entry["passed"] else "FAIL"
        detail = f"  ({entry['detail']})" if entry["detail"] else ""
        print(f"  [{mark}] {entry['check']}{detail}")
    print("-" * 78)
    raise InputContractError(
        "The stage-3 inputs are incomplete (Plan B). Nothing was computed and nothing was "
        "written. Missing: "
        + (f"directories {missing_directories}; " if missing_directories else "")
        + (f"files {missing_files}" if missing_files else "")
        + ". Place the stage-1 and stage-2 aggregate files and the raw course CSV folder in "
        "the configured directories, then run the notebook again."
    )

# --- 2. read the stage-1 artefacts ------------------------------------------
s1_inventory = pd.read_csv(stage1_dir / STAGE1_INVENTORY)
s1_metrics = pd.read_csv(stage1_dir / STAGE1_METRICS)
with open(stage1_dir / STAGE1_METADATA, "r", encoding="utf-8") as handle:
    s1_metadata = json.load(handle)

for name, frame in ((STAGE1_INVENTORY, s1_inventory), (STAGE1_METRICS, s1_metrics)):
    check(f"{name} is not empty", len(frame) > 0, f"{len(frame)} row(s)")
    missing_columns = [c for c in REQUIRED_COLUMNS[name] if c not in frame.columns]
    check(
        f"{name} has every required column",
        not missing_columns,
        f"missing: {missing_columns}",
    )
    if "results_validity" in frame.columns:
        stamps = set(frame["results_validity"].dropna().astype(str))
        check(
            f"{name}: results_validity is a recognised stamp",
            stamps.issubset(ALLOWED_VALIDITY),
            f"found {sorted(stamps)}",
        )

check(
    f"{STAGE1_METADATA}: run_status is 'completed'",
    str(s1_metadata.get("run_status", "")) == REQUIRED_RUN_STATUS,
    f"found {s1_metadata.get('run_status')!r}",
)
check(
    f"{STAGE1_METADATA}: cohort_mode is 'complete_case_all_candidates'",
    str(s1_metadata.get("cohort_mode", "")) == REQUIRED_COHORT_MODE,
    f"found {s1_metadata.get('cohort_mode')!r}",
)
check(
    f"{STAGE1_METADATA}: results_validity is a recognised stamp",
    str(s1_metadata.get("results_validity", "")) in ALLOWED_VALIDITY,
    f"found {s1_metadata.get('results_validity')!r}",
)

duplicates = s1_metrics[s1_metrics.duplicated(subset=["feature", "model"], keep=False)]
check(
    f"{STAGE1_METRICS}: no duplicated (feature, model) row",
    len(duplicates) == 0,
    f"{len(duplicates)} duplicated row(s)",
)
unknown_models = sorted(set(s1_metrics["model"].astype(str)) - KNOWN_MODELS)
check(
    f"{STAGE1_METRICS}: every model is a known stage-1 model",
    not unknown_models,
    f"unknown: {unknown_models}",
)
missing_models = sorted(KNOWN_MODELS - set(s1_metrics["model"].astype(str)))
check(
    f"{STAGE1_METRICS}: the three stage-1 models are present",
    not missing_models,
    f"missing: {missing_models}",
)

stage1_candidate_columns = sorted(
    s1_inventory.loc[s1_inventory["status"].astype(str) == "candidate", "column"].astype(str).tolist()
)
check(
    f"{STAGE1_INVENTORY}: the candidate pool is not empty",
    len(stage1_candidate_columns) > 0,
    f"{len(stage1_candidate_columns)} candidate column(s)",
)
check(
    f"{STAGE1_INVENTORY}: the label column is not a candidate",
    LABEL_COLUMN not in stage1_candidate_columns,
    "the label must never be a feature",
)

# --- 3. read the stage-2 artefacts ------------------------------------------
s2_selected_all = pd.read_csv(stage2_dir / STAGE2_SELECTED)
with open(stage2_dir / STAGE2_MANIFEST, "r", encoding="utf-8") as handle:
    s2_manifest = json.load(handle)

check(
    f"{STAGE2_SELECTED} is not empty",
    len(s2_selected_all) > 0,
    f"{len(s2_selected_all)} row(s)",
)
missing_columns = [c for c in REQUIRED_COLUMNS[STAGE2_SELECTED] if c not in s2_selected_all.columns]
check(
    f"{STAGE2_SELECTED} has every required column",
    not missing_columns,
    f"missing: {missing_columns}",
)
if "results_validity" in s2_selected_all.columns:
    stamps = set(s2_selected_all["results_validity"].dropna().astype(str))
    check(
        f"{STAGE2_SELECTED}: results_validity is a recognised stamp",
        stamps.issubset(ALLOWED_VALIDITY),
        f"found {sorted(stamps)}",
    )

check(
    f"{STAGE2_MANIFEST}: results_validity is a recognised stamp",
    str(s2_manifest.get("results_validity", "")) in ALLOWED_VALIDITY,
    f"found {s2_manifest.get('results_validity')!r}",
)
stage2_validation = s2_manifest.get("validation", {})
check(
    f"{STAGE2_MANIFEST}: the stage-2 run passed its own input contract",
    bool(stage2_validation.get("all_checks_passed", False)),
    f"all_checks_passed={stage2_validation.get('all_checks_passed')!r}",
)

s2_selected = s2_selected_all[s2_selected_all["selected"].astype(str).str.lower() == "true"].copy()
check(
    f"{STAGE2_SELECTED}: the shortlist is not empty",
    len(s2_selected) > 0,
    f"{len(s2_selected)} shortlisted feature(s)",
)
check(
    f"{STAGE2_SELECTED}: no duplicated feature row",
    not s2_selected["feature"].astype(str).duplicated().any(),
    "the shortlist must hold each feature once",
)

shortlisted_features = sorted(s2_selected["feature"].astype(str).tolist())

# --- 4. the shortlist must agree with stage 1 -------------------------------
metric_features = set(s1_metrics["feature"].astype(str))
unknown_to_stage1 = [f for f in shortlisted_features if f not in metric_features]
check(
    "every shortlisted feature was screened in stage 1",
    not unknown_to_stage1,
    f"unknown to stage 1: {unknown_to_stage1}",
)
not_candidates = [f for f in shortlisted_features if f not in stage1_candidate_columns]
check(
    "every shortlisted feature is a stage-1 candidate column",
    not not_candidates,
    f"not a candidate: {not_candidates}",
)

# --- 5. one split, everywhere -------------------------------------------------
stage1_fingerprint = str(s1_metadata.get("split_fingerprint", ""))
metrics_fingerprints = sorted(set(s1_metrics["split_fingerprint"].dropna().astype(str)))
selected_fingerprints = sorted(set(s2_selected["split_fingerprint"].dropna().astype(str)))
manifest_fingerprint = str(s2_manifest.get("inputs", {}).get("split_fingerprint", ""))

check(
    "stage-1 metrics hold exactly one split fingerprint",
    len(metrics_fingerprints) == 1,
    f"found {metrics_fingerprints}",
)
check(
    "stage-2 shortlist holds exactly one split fingerprint",
    len(selected_fingerprints) == 1,
    f"found {selected_fingerprints}",
)
check(
    "the stage-1 and stage-2 split fingerprints agree",
    bool(stage1_fingerprint)
    and stage1_fingerprint == metrics_fingerprints[0] == selected_fingerprints[0] == manifest_fingerprint,
    f"stage1={stage1_fingerprint!r} metrics={metrics_fingerprints} "
    f"stage2={selected_fingerprints} manifest={manifest_fingerprint!r}",
)

# --- 6. report ---------------------------------------------------------------
print("INPUT CONTRACT REPORT")
print("-" * 78)
for entry in validation_results:
    mark = "PASS" if entry["passed"] else "FAIL"
    detail = f"  ({entry['detail']})" if entry["detail"] else ""
    print(f"  [{mark}] {entry['check']}{detail}")
print("-" * 78)
failed_checks = [e for e in validation_results if not e["passed"]]
if failed_checks:
    raise InputContractError(
        f"{len(failed_checks)} input-contract check(s) failed: "
        f"{[e['check'] for e in failed_checks]}. The run stops here; no correlation, no feature "
        "set and no conclusion is produced."
    )
print(f"All {len(validation_results)} input-contract checks passed.")
print()
print(f"stage-1 candidate columns : {len(stage1_candidate_columns)}")
print(f"stage-2 shortlisted        : {len(shortlisted_features)}")
print(f"split fingerprint          : {stage1_fingerprint}")


INPUT CONTRACT REPORT
------------------------------------------------------------------------------
  [PASS] the three input directories exist  (missing: [])
  [PASS] all five aggregate input files exist  (missing: [])
  [PASS] feature_inventory.csv is not empty  (70 row(s))
  [PASS] feature_inventory.csv has every required column  (missing: [])
  [PASS] feature_inventory.csv: results_validity is a recognised stamp  (found ['only valid after a real Colab run'])
  [PASS] single_feature_metrics.csv is not empty  (171 row(s))
  [PASS] single_feature_metrics.csv has every required column  (missing: [])
  [PASS] single_feature_metrics.csv: results_validity is a recognised stamp  (found ['only valid after a real Colab run'])
  [PASS] run_metadata.json: run_status is 'completed'  (found 'completed')
  [PASS] run_metadata.json: cohort_mode is 'complete_case_all_candidates'  (found 'complete_case_all_candidates')
  [PASS] run_metadata.json: results_validity is a recognised stamp  (found 'only 

In [18]:
# =============================================================================
#  STEP 2 — REBUILD THE STAGE-1 COHORT FROM THE RAW DATA
#  The cohort must be the same complete-case cohort stage 1 used, otherwise the
#  correlations would not be comparable with the stage-1 metrics. The rebuild is
#  therefore cross-checked against the stage-1 manifest, and a mismatch is an
#  explicit failure — never a silent continuation.
# =============================================================================
raw_files = sorted(p for p in raw_dir.glob(FILE_PATTERN) if p.is_file())
require(
    raw_files,
    f"no raw CSV file matching {FILE_PATTERN!r} was found in {raw_dir}. The correlation needs "
    "the course data in the Colab runtime; it is never committed to the repository.",
)

frames = []
skipped_raw_files = []
for path in raw_files:
    try:
        chunk = pd.read_csv(path)
    except Exception:
        # a dirty numeric column makes pandas raise; re-read as text and coerce later
        chunk = pd.read_csv(path, dtype=str)
    if LABEL_COLUMN not in chunk.columns:
        skipped_raw_files.append(f"{path.name}: no {LABEL_COLUMN!r} column")
        continue
    frames.append(chunk)

require(
    frames,
    "no usable raw file was loaded. The label is never derived from a file name, so a file "
    f"without the {LABEL_COLUMN!r} column cannot contribute rows. Skipped: {skipped_raw_files}",
)

raw_frame = pd.concat(frames, ignore_index=True, sort=False)
del frames

# Create the aggregate-only destination before any reconstruction failure can
# be recorded. This prevents a secondary FileNotFoundError from hiding the
# actual cohort-contract error.
output_dir = Path(OUTPUT_DIR).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

# --- the shortlisted features must exist in the raw data ----------------------
raw_columns = set(raw_frame.columns.astype(str))
absent_in_raw = [f for f in shortlisted_features if f not in raw_columns]
require(
    not absent_in_raw,
    f"these shortlisted features are absent from the raw CSV files: {absent_in_raw}. The "
    "correlations cannot be computed for them.",
)
absent_candidates = [c for c in stage1_candidate_columns if c not in raw_columns]
require(
    not absent_candidates,
    f"stage-1 candidate columns are absent from the raw CSV files: {absent_candidates}. "
    "The complete-case cohort cannot be reproduced when any candidate column is missing.",
)

# --- rebuild the complete-case cohort ----------------------------------------
label_ok = raw_frame[LABEL_COLUMN].notna()
class_counts = raw_frame.loc[label_ok, LABEL_COLUMN].value_counts()
kept_classes = class_counts[class_counts >= MIN_CLASS_COUNT].index
class_ok = raw_frame[LABEL_COLUMN].isin(kept_classes)

complete_case = pd.Series(True, index=raw_frame.index)
numeric_candidates = pd.DataFrame(index=raw_frame.index)
for column in stage1_candidate_columns:
    values, finite = numeric_finite_mask(raw_frame[column])
    numeric_candidates[column] = values
    complete_case &= finite

cohort_mask = label_ok & class_ok & complete_case
# Correlations must use the same safely numeric values that were used to
# define completeness. Retaining raw text here could make Pearson fail or
# make Spearman rank strings lexicographically.
cohort = numeric_candidates.loc[cohort_mask, stage1_candidate_columns].copy()
cohort[LABEL_COLUMN] = raw_frame.loc[cohort_mask, LABEL_COLUMN].to_numpy()
cohort = cohort.reset_index(drop=True)

reconstructed_rows = int(len(cohort))
reconstructed_classes = int(cohort[LABEL_COLUMN].nunique())

# --- cross-check against the stage-1 manifest --------------------------------
s1_config = s1_metadata.get("configuration", {})
reconstruction_checks = [
    {
        "check": "n_rows_common_cohort",
        "stage1": s1_metadata.get("n_rows_common_cohort"),
        "reconstructed": reconstructed_rows,
        "passed": s1_metadata.get("n_rows_common_cohort") == reconstructed_rows,
    },
    {
        "check": "n_classes_common",
        "stage1": s1_metadata.get("n_classes_common"),
        "reconstructed": reconstructed_classes,
        "passed": s1_metadata.get("n_classes_common") == reconstructed_classes,
    },
    {
        "check": "MIN_CLASS_COUNT",
        "stage1": s1_config.get("MIN_CLASS_COUNT"),
        "reconstructed": MIN_CLASS_COUNT,
        "passed": s1_config.get("MIN_CLASS_COUNT") == MIN_CLASS_COUNT,
    },
    {
        "check": "RANDOM_STATE",
        "stage1": s1_config.get("RANDOM_STATE"),
        "reconstructed": RANDOM_STATE,
        "passed": s1_config.get("RANDOM_STATE") == RANDOM_STATE,
    },
    {
        "check": "TEST_SIZE",
        "stage1": s1_config.get("TEST_SIZE"),
        "reconstructed": TEST_SIZE,
        "passed": s1_config.get("TEST_SIZE") == TEST_SIZE,
    },
]
reconstruction_failures = [c for c in reconstruction_checks if not c["passed"]]

if reconstruction_failures:
    failure_detail = "; ".join(
        f"{c['check']}: stage1={c['stage1']!r} vs reconstructed={c['reconstructed']!r}"
        for c in reconstruction_failures
    )
    # the failure is recorded before the run stops, so it is never lost
    dump_json(
        Path(OUTPUT_DIR) / "correlation_summary.json",
        {
            "results_validity": RESULTS_VALIDITY,
            "stage": "V3 stage 3 - feature correlation and candidate feature-set definition",
            "run_timestamp_utc": utc_now_iso(),
            "reconstruction_status": "failed",
            "reconstruction_checks": reconstruction_checks,
            "failure": failure_detail,
            "conclusion": "No correlation, no feature set and no conclusion was produced.",
        },
    )
    raise ReconstructionError(
        "the rebuilt cohort does not match the stage-1 manifest, so the correlations would not "
        f"be comparable with the stage-1 metrics. Details: {failure_detail}. Nothing was "
        "computed. Align RAW_DATA_DIR / MIN_CLASS_COUNT / the stage-1 inputs and run again."
    )

print(f"raw files matched          : {len(raw_files)}")
print(f"raw files used             : {len(raw_files) - len(skipped_raw_files)}")
print(f"rows merged                : {len(raw_frame)}")
print(f"rebuilt cohort rows        : {reconstructed_rows}")
print(f"rebuilt cohort classes     : {reconstructed_classes}")
print(f"stage-1 candidate columns  : {len(stage1_candidate_columns)}")
print()
print("Reconstruction cross-check against the stage-1 manifest:")
for entry in reconstruction_checks:
    mark = "PASS" if entry["passed"] else "FAIL"
    print(f"  [{mark}] {entry['check']}: stage1={entry['stage1']!r} reconstructed={entry['reconstructed']!r}")
print()
print("The cohort is the same complete-case cohort stage 1 used, so the correlations are")
print("comparable with the stage-1 metrics.")


raw files matched          : 10
raw files used             : 10
rows merged                : 2002
rebuilt cohort rows        : 2002
rebuilt cohort classes     : 9
stage-1 candidate columns  : 57

Reconstruction cross-check against the stage-1 manifest:
  [PASS] n_rows_common_cohort: stage1=2002 reconstructed=2002
  [PASS] n_classes_common: stage1=9 reconstructed=9
  [PASS] MIN_CLASS_COUNT: stage1=5 reconstructed=5
  [PASS] RANDOM_STATE: stage1=42 reconstructed=42
  [PASS] TEST_SIZE: stage1=0.2 reconstructed=0.2

The cohort is the same complete-case cohort stage 1 used, so the correlations are
comparable with the stage-1 metrics.


In [19]:
# =============================================================================
#  STEP 3 — FEATURE-TO-FEATURE CORRELATION
#  Computed on the rebuilt cohort, for the stage-2 shortlisted features only.
#  Pearson and Spearman are computed and reported separately; neither is a
#  model-ranking agreement, and neither is a performance measure.
# =============================================================================
shortlist = sorted(shortlisted_features)

correlation_frame = cohort[shortlist]
constant_features = [c for c in shortlist if correlation_frame[c].nunique(dropna=True) <= 1]
if constant_features:
    print(f"note: constant inside the cohort (no defined correlation): {constant_features}")

pearson_matrix = correlation_frame.corr(method="pearson")
spearman_matrix_values = spearman_matrix(correlation_frame)

pearson_matrix.index.name = "feature"
spearman_matrix_values.index.name = "feature"

# --- the pairs table (upper triangle only) ------------------------------------
pair_rows = []
for i, feature_a in enumerate(shortlist):
    for feature_b in shortlist[i + 1:]:
        pearson_value = float(pearson_matrix.loc[feature_a, feature_b])
        spearman_value = float(spearman_matrix_values.loc[feature_a, feature_b])
        abs_pearson = abs(pearson_value)
        abs_spearman = abs(spearman_value)
        redundant_pearson = bool(np.isfinite(abs_pearson) and abs_pearson >= CORRELATION_THRESHOLD)
        redundant_spearman = bool(np.isfinite(abs_spearman) and abs_spearman >= CORRELATION_THRESHOLD)
        pair_rows.append(
            {
                "feature_a": feature_a,
                "feature_b": feature_b,
                "pearson": pearson_value,
                "spearman": spearman_value,
                "abs_pearson": abs_pearson,
                "abs_spearman": abs_spearman,
                "redundant_pearson": redundant_pearson,
                "redundant_spearman": redundant_spearman,
                "redundant_either": redundant_pearson or redundant_spearman,
                "correlation_defined": bool(np.isfinite(pearson_value) and np.isfinite(spearman_value)),
                "potential_recording_or_session_risk_a": is_absolute_statistic(feature_a),
                "potential_recording_or_session_risk_b": is_absolute_statistic(feature_b),
                "potential_recording_or_session_risk_either": (
                    is_absolute_statistic(feature_a) or is_absolute_statistic(feature_b)
                ),
            }
        )

correlation_pairs = pd.DataFrame(pair_rows)
correlation_pairs = correlation_pairs.sort_values(
    ["redundant_either", "abs_pearson", "feature_a", "feature_b"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

n_pairs = int(len(correlation_pairs))
n_redundant_pairs = int(correlation_pairs["redundant_either"].sum()) if n_pairs else 0
max_abs_pearson = float(np.nanmax(np.abs(pearson_matrix.to_numpy()))) if len(shortlist) > 1 else 0.0
max_abs_spearman = float(np.nanmax(np.abs(spearman_matrix_values.to_numpy()))) if len(shortlist) > 1 else 0.0

# --- write the artefacts -----------------------------------------------------
# The matrix files carry a constant stamp column, so the validity notice travels
# with the file. Drop that column before using the matrix numerically.
pearson_out = stamp(pearson_matrix.reset_index())
spearman_out = stamp(spearman_matrix_values.reset_index())
pearson_path = output_dir / "pearson_correlation.csv"
spearman_path = output_dir / "spearman_correlation.csv"
pearson_out.to_csv(pearson_path, index=False)
spearman_out.to_csv(spearman_path, index=False)

pairs_path = output_dir / "correlation_pairs.csv"
stamp(correlation_pairs).to_csv(pairs_path, index=False)

print(f"features correlated : {len(shortlist)}")
print(f"pairs (upper triangle): {n_pairs}")
print(f"pairs at or above the redundancy threshold ({CORRELATION_THRESHOLD}): {n_redundant_pairs}")
print(f"max |Pearson|  : {max_abs_pearson:.4f}")
print(f"max |Spearman| : {max_abs_spearman:.4f}")
print()
if n_redundant_pairs:
    print("Most redundant pairs:")
    for _, row in correlation_pairs[correlation_pairs["redundant_either"]].head(10).iterrows():
        print(
            f"  {row['feature_a']:<12s} {row['feature_b']:<12s} "
            f"pearson={row['pearson']:+.4f} spearman={row['spearman']:+.4f}"
        )
print()
print("written:")
for path in (pearson_path, spearman_path, pairs_path):
    print(f"  {path.name} ({path.stat().st_size} bytes)")


features correlated : 12
pairs (upper triangle): 66
pairs at or above the redundancy threshold (0.85): 9
max |Pearson|  : 1.0000
max |Spearman| : 1.0000

Most redundant pairs:
  avg_t        max_t        pearson=+0.9979 spearman=+0.9991
  avg_y        max_y        pearson=+0.9974 spearman=+0.9974
  avg_x        max_x        pearson=+0.9973 spearman=+0.9966
  avg_t        min_t        pearson=+0.9973 spearman=+0.9911
  avg_x        min_x        pearson=+0.9971 spearman=+0.9974
  avg_y        min_y        pearson=+0.9961 spearman=+0.9981
  max_t        min_t        pearson=+0.9914 spearman=+0.9853
  max_x        min_x        pearson=+0.9893 spearman=+0.9894
  max_y        min_y        pearson=+0.9878 spearman=+0.9922

written:
  pearson_correlation.csv (3293 bytes)
  spearman_correlation.csv (3261 bytes)
  correlation_pairs.csv (11072 bytes)


In [20]:
# =============================================================================
#  STEP 4 — FIGURES
#  Two correlation heatmaps and one redundancy audit chart. All of them are
#  drawn from the computed matrices; nothing is hard-coded.
# =============================================================================
saved_figures = []


def save_figure(fig, filename):
    if SAVE_FIGURES:
        figures_dir = output_dir / "figures"
        figures_dir.mkdir(parents=True, exist_ok=True)
        path = figures_dir / filename
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
        saved_figures.append(filename)
        print(f"  saved: figures/{filename}")
    plt.close(fig)


def draw_heatmap(matrix, title, filename, note=""):
    """Correlation heatmap with the coefficients printed in every cell."""
    size = max(6.0, 0.62 * len(shortlist) + 2.6)
    fig, ax = plt.subplots(figsize=(size, size))
    values = matrix.to_numpy(dtype="float64")
    masked = np.ma.masked_invalid(values)
    image = ax.imshow(masked, cmap="RdBu_r", vmin=-1.0, vmax=1.0, aspect="auto")

    ax.set_xticks(np.arange(len(shortlist)))
    ax.set_yticks(np.arange(len(shortlist)))
    ax.set_xticklabels(shortlist, rotation=90, fontsize=8)
    ax.set_yticklabels(shortlist, fontsize=8)
    ax.set_title(title + (f"\n{note}" if note else ""), fontsize=10)

    threshold = CORRELATION_THRESHOLD
    for row_index in range(len(shortlist)):
        for column_index in range(len(shortlist)):
            value = values[row_index, column_index]
            if not np.isfinite(value):
                text, colour = "n/a", "#888888"
            else:
                text = f"{value:.2f}"
                colour = "#ffffff" if abs(value) > 0.55 else "#222222"
                if row_index != column_index and abs(value) >= threshold:
                    text += "*"
                    colour = "#7a1414" if abs(value) < 0.85 else "#ffffff"
            ax.text(
                column_index, row_index, text,
                ha="center", va="center", fontsize=6.5, color=colour,
            )

    bar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
    bar.set_label("correlation coefficient", fontsize=8)
    bar.ax.tick_params(labelsize=7)
    fig.tight_layout()
    save_figure(fig, filename)


draw_heatmap(
    pearson_matrix,
    f"Pearson correlation between shortlisted features (n = {len(shortlist)})",
    "pearson_heatmap.png",
    note=f"* marks |r| >= {CORRELATION_THRESHOLD}; computed on the stage-1 cohort",
)
draw_heatmap(
    spearman_matrix_values,
    f"Spearman correlation between shortlisted features (n = {len(shortlist)})",
    "spearman_heatmap.png",
    note=f"* marks |rho| >= {CORRELATION_THRESHOLD}; computed on the stage-1 cohort",
)

# --- the most redundant pairs -------------------------------------------------
if n_pairs:
    chart = correlation_pairs.copy()
    chart["max_abs"] = chart[["abs_pearson", "abs_spearman"]].max(axis=1)
    chart = chart.sort_values("max_abs", ascending=False).head(15).iloc[::-1]
    labels = [f"{a} / {b}" for a, b in zip(chart["feature_a"], chart["feature_b"])]
    positions = np.arange(len(chart))

    fig, ax = plt.subplots(figsize=(8.6, max(3.2, 0.34 * len(chart) + 1.4)))
    ax.barh(positions - 0.19, chart["abs_pearson"], height=0.36, color="#2f6fb2",
            alpha=0.85, label="|Pearson|")
    ax.barh(positions + 0.19, chart["abs_spearman"], height=0.36, color="#c47f4a",
            alpha=0.85, label="|Spearman|")
    ax.axvline(CORRELATION_THRESHOLD, color="#b03a3a", linestyle="--", linewidth=1.1,
               label=f"redundancy threshold ({CORRELATION_THRESHOLD})")
    ax.set_yticks(positions)
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel("absolute correlation coefficient")
    ax.set_title(
        "Most redundant pairs among the shortlisted features\n"
        f"{n_redundant_pairs} of {n_pairs} pairs reach the threshold",
        fontsize=10,
    )
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    ax.grid(True, axis="x", alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    save_figure(fig, "redundant_pairs.png")

print()
print(f"figures written: {len(saved_figures)}")


  saved: figures/pearson_heatmap.png
  saved: figures/spearman_heatmap.png
  saved: figures/redundant_pairs.png

figures written: 3


In [21]:
# =============================================================================
#  STEP 5 — THREE CANDIDATE FEATURE SETS
#  Deterministic, low-redundancy and auditable. No model is trained, no set is
#  scored, and no set is announced as the champion: the sets are candidates to
#  be evaluated in the next stage.
# =============================================================================
s2_selected = s2_selected.copy()
s2_selected["_max_macro_f1"] = s2_selected[
    ["macro_f1_rbf_svc", "macro_f1_random_forest"]
].max(axis=1)

feature_set_definitions = {}
membership_rows = []

for set_name, (rank_column, rule_text) in FEATURE_SET_RULES.items():
    ordered = s2_selected.sort_values(
        [rank_column, "_max_macro_f1", "feature"], ascending=[True, False, True]
    )

    accepted = []
    accepted_details = []
    rejected_details = []

    for order_position, (_, row) in enumerate(ordered.iterrows(), start=1):
        feature = str(row["feature"])
        entry = {
            "feature": feature,
            "order_position": order_position,
            "ordering_rank": float(row[rank_column]),
            "mean_rank": float(row["mean_rank"]),
            "rank_rbf_svc": float(row["rank_rbf_svc"]),
            "rank_random_forest": float(row["rank_random_forest"]),
            "macro_f1_rbf_svc": float(row["macro_f1_rbf_svc"]),
            "macro_f1_random_forest": float(row["macro_f1_random_forest"]),
            "potential_recording_or_session_risk": is_absolute_statistic(feature),
        }

        if feature in constant_features:
            entry["rejection_reason"] = "constant_in_cohort"
            entry["max_abs_pearson_vs_set"] = None
            entry["max_abs_spearman_vs_set"] = None
            rejected_details.append(entry)
            continue

        blocking_feature = None
        blocking_pearson = None
        blocking_spearman = None
        max_abs_pearson = 0.0
        max_abs_spearman = 0.0
        for member in accepted:
            pearson_value = abs(float(pearson_matrix.loc[feature, member]))
            spearman_value = abs(float(spearman_matrix_values.loc[feature, member]))
            max_abs_pearson = max(max_abs_pearson, pearson_value)
            max_abs_spearman = max(max_abs_spearman, spearman_value)
            if (
                pearson_value >= CORRELATION_THRESHOLD
                or spearman_value >= CORRELATION_THRESHOLD
            ):
                blocking_feature = member
                blocking_pearson = pearson_value
                blocking_spearman = spearman_value
                break

        entry["max_abs_pearson_vs_set"] = round(max_abs_pearson, 6)
        entry["max_abs_spearman_vs_set"] = round(max_abs_spearman, 6)

        # the redundancy test is applied first, so the recorded reason is the most
        # informative one even when the set is already full
        if blocking_feature is not None:
            entry["rejection_reason"] = (
                f"redundant_with:{blocking_feature} "
                f"(|pearson|={blocking_pearson:.4f}, |spearman|={blocking_spearman:.4f} "
                f">= {CORRELATION_THRESHOLD})"
            )
            rejected_details.append(entry)
            continue

        if len(accepted) >= MAX_FEATURES_PER_SET:
            entry["rejection_reason"] = "set_full"
            rejected_details.append(entry)
            continue

        entry["position"] = len(accepted) + 1
        entry["rejection_reason"] = ""
        accepted.append(feature)
        accepted_details.append(entry)

    if len(accepted) >= 2:
        set_status = SET_STATUS_PROVISIONAL
    else:
        set_status = SET_STATUS_INSUFFICIENT

    feature_set_definitions[set_name] = {
        "set_name": set_name,
        "status": set_status,
        "evaluated": False,
        "scored": False,
        "ordering_rule": rule_text,
        "ordering_column": rank_column,
        "tie_break": "the larger of the two models' macro F1 (descending), then feature name (ascending)",
        "acceptance_rule": (
            "a feature is accepted only when its |Pearson| and its |Spearman| against every "
            f"already accepted feature are both below {CORRELATION_THRESHOLD}"
        ),
        "correlation_threshold": CORRELATION_THRESHOLD,
        "max_features_per_set": MAX_FEATURES_PER_SET,
        "n_features": len(accepted),
        "features": accepted,
        "members": accepted_details,
        "rejected": rejected_details,
        "contains_potential_recording_or_session_risk": any(
            m["potential_recording_or_session_risk"] for m in accepted_details
        ),
        "note": (
            "A candidate set for a later evaluation stage. It has not been scored, and nothing "
            "here claims which set is best."
        ),
    }

    for entry in accepted_details:
        membership_rows.append(
            {
                "set_name": set_name,
                "feature": entry["feature"],
                "in_set": True,
                "position": entry["position"],
                "order_position": entry["order_position"],
                "ordering_rank": entry["ordering_rank"],
                "mean_rank": entry["mean_rank"],
                "rank_rbf_svc": entry["rank_rbf_svc"],
                "rank_random_forest": entry["rank_random_forest"],
                "macro_f1_rbf_svc": entry["macro_f1_rbf_svc"],
                "macro_f1_random_forest": entry["macro_f1_random_forest"],
                "max_abs_pearson_vs_set": entry["max_abs_pearson_vs_set"],
                "max_abs_spearman_vs_set": entry["max_abs_spearman_vs_set"],
                "rejection_reason": "",
                "potential_recording_or_session_risk": entry["potential_recording_or_session_risk"],
                "set_status": set_status,
                "results_validity": RESULTS_VALIDITY,
                "split_fingerprint": stage1_fingerprint,
            }
        )
    for entry in rejected_details:
        membership_rows.append(
            {
                "set_name": set_name,
                "feature": entry["feature"],
                "in_set": False,
                "position": "",
                "order_position": entry["order_position"],
                "ordering_rank": entry["ordering_rank"],
                "mean_rank": entry["mean_rank"],
                "rank_rbf_svc": entry["rank_rbf_svc"],
                "rank_random_forest": entry["rank_random_forest"],
                "macro_f1_rbf_svc": entry["macro_f1_rbf_svc"],
                "macro_f1_random_forest": entry["macro_f1_random_forest"],
                "max_abs_pearson_vs_set": entry["max_abs_pearson_vs_set"],
                "max_abs_spearman_vs_set": entry["max_abs_spearman_vs_set"],
                "rejection_reason": entry["rejection_reason"],
                "potential_recording_or_session_risk": entry["potential_recording_or_session_risk"],
                "set_status": set_status,
                "results_validity": RESULTS_VALIDITY,
                "split_fingerprint": stage1_fingerprint,
            }
        )

feature_set_membership = pd.DataFrame(membership_rows)

for set_name, definition in feature_set_definitions.items():
    print(f"{set_name}  [{definition['status']}]  n = {definition['n_features']}")
    for member in definition["members"]:
        flag = "  <-- potential recording/session risk" if member[
            "potential_recording_or_session_risk"
        ] else ""
        print(
            f"    {member['position']:>2d}. {member['feature']:<12s} "
            f"order_rank={member['ordering_rank']:.1f} "
            f"max|r| vs set <= {member['max_abs_pearson_vs_set']:.3f}{flag}"
        )
    print()

incomplete_sets = [
    name for name, definition in feature_set_definitions.items()
    if definition["status"] == SET_STATUS_INSUFFICIENT
]
if incomplete_sets:
    print("These sets could not be built from non-redundant candidates:")
    for name in incomplete_sets:
        print(f"  - {name}: {SET_STATUS_INSUFFICIENT}")
    print("They are reported as such; no feature was substituted to make them look complete.")


FS1_consensus_compact  [provisional / not yet evaluated]  n = 6
     1. max_y        order_rank=3.0 max|r| vs set <= 0.000  <-- potential recording/session risk
     2. min_t        order_rank=6.0 max|r| vs set <= 0.245  <-- potential recording/session risk
     3. min_x        order_rank=10.0 max|r| vs set <= 0.149  <-- potential recording/session risk
     4. VV2          order_rank=11.0 max|r| vs set <= 0.091
     5. VV0          order_rank=14.5 max|r| vs set <= 0.695
     6. min_hv       order_rank=15.0 max|r| vs set <= 0.231

FS2_svc_diverse  [provisional / not yet evaluated]  n = 6
     1. avg_t        order_rank=1.0 max|r| vs set <= 0.000  <-- potential recording/session risk
     2. max_y        order_rank=4.0 max|r| vs set <= 0.241  <-- potential recording/session risk
     3. min_x        order_rank=7.0 max|r| vs set <= 0.149  <-- potential recording/session risk
     4. VV2          order_rank=17.0 max|r| vs set <= 0.091
     5. min_hv       order_rank=21.0 max|r| vs set <= 

In [22]:
# =============================================================================
#  STEP 6 — WRITE THE MANIFEST AND THE FEATURE-SET DEFINITIONS
#  Aggregate only: no raw rows, no per-row values, no label values, no source
#  file names, no absolute paths.
# =============================================================================
input_hashes = {
    f"stage1/{name}": sha256_file(stage1_dir / name) for name in REQUIRED_STAGE1_FILES
}
input_hashes.update(
    {f"stage2/{name}": sha256_file(stage2_dir / name) for name in REQUIRED_STAGE2_FILES}
)
# A digest of the raw file set, not the names themselves: the names are
# participant-linked and must not appear in any artefact.
raw_file_digest = sha256_text("\n".join(path.name for path in raw_files))

correlation_summary = {
    "results_validity": RESULTS_VALIDITY,
    "notice": (
        "Produced by a real Colab run over the stage-1 and stage-2 aggregate outputs and the "
        "raw course data read in the runtime. No model was trained, no feature set was scored "
        "and no champion was selected."
    ),
    "stage": "V3 stage 3 - feature correlation and candidate feature-set definition",
    "run_timestamp_utc": utc_now_iso(),
    "configuration": {
        "RAW_DATA_DIR": "<withheld: absolute local path is not recorded>",
        "FILE_PATTERN": FILE_PATTERN,
        "LABEL_COLUMN": LABEL_COLUMN,
        "STAGE1_OUTPUT_DIR": "<withheld: absolute local path is not recorded>",
        "STAGE2_OUTPUT_DIR": "<withheld: absolute local path is not recorded>",
        "OUTPUT_DIR": "<withheld: absolute local path is not recorded>",
        "MIN_CLASS_COUNT": MIN_CLASS_COUNT,
        "CORRELATION_THRESHOLD": CORRELATION_THRESHOLD,
        "MAX_FEATURES_PER_SET": MAX_FEATURES_PER_SET,
        "RANDOM_STATE": RANDOM_STATE,
        "TEST_SIZE": TEST_SIZE,
        "SAVE_FIGURES": SAVE_FIGURES,
        "FIGURE_DPI": FIGURE_DPI,
    },
    "inputs": {
        "files": input_hashes,
        "raw_file_set_sha256": raw_file_digest,
        "n_raw_files_matched": len(raw_files),
        "n_raw_files_used": len(raw_files) - len(skipped_raw_files),
        "split_fingerprint": stage1_fingerprint,
    },
    "validation": {
        "all_checks_passed": all(entry["passed"] for entry in validation_results),
        "n_checks": len(validation_results),
        "checks": validation_results,
        "reconstruction_checks": reconstruction_checks,
        "reconstruction_status": "passed" if not reconstruction_failures else "failed",
    },
    "cohort": {
        "n_rows_merged": int(len(raw_frame)),
        "n_rows_common_cohort": reconstructed_rows,
        "n_classes_common": reconstructed_classes,
        "n_candidate_columns": len(stage1_candidate_columns),
        "n_shortlisted_features": len(shortlist),
        "note": (
            "The cohort is rebuilt from the raw data with the stage-1 complete-case rule and "
            "cross-checked against the stage-1 manifest."
        ),
    },
    "correlation": {
        "n_features": len(shortlist),
        "n_pairs": n_pairs,
        "n_redundant_pairs": n_redundant_pairs,
        "threshold": CORRELATION_THRESHOLD,
        "max_abs_pearson": max_abs_pearson,
        "max_abs_spearman": max_abs_spearman,
        "constant_in_cohort": constant_features,
        "coefficients": "Pearson and Spearman, computed separately on the cohort",
        "n_potential_recording_or_session_risk_features": int(
            sum(1 for f in shortlist if is_absolute_statistic(f))
        ),
    },
    "feature_sets": {
        name: {
            "status": definition["status"],
            "n_features": definition["n_features"],
            "features": definition["features"],
            "ordering_rule": definition["ordering_rule"],
        }
        for name, definition in feature_set_definitions.items()
    },
    "figures": sorted(saved_figures),
    "limitations": [
        "No model was trained, no feature set was scored and no champion was selected; the sets are provisional candidates for a later stage.",
        "A correlation between two features says nothing about how useful either is for identification; redundancy is a constraint on the set, not evidence of information.",
        "The cohort is a complete-case subsample of the merged data and the rows it drops are not missing at random.",
        "The correlations are computed on one cohort and inherit the stage-1 split uncertainty; no bootstrap interval is reported.",
        "Features flagged potential_recording_or_session_risk are absolute position or time statistics and may encode the recording rather than the person.",
        "The label is taken from the CSV content only and is never derived from a file name.",
        "No raw data, per-row value, label value, source file name or absolute path is written.",
    ],
}

summary_path = dump_json(output_dir / "correlation_summary.json", correlation_summary)

definitions_payload = {
    "results_validity": RESULTS_VALIDITY,
    "notice": (
        "These are candidate feature sets for a later evaluation stage. They have not been "
        "scored, and no set is presented as better than another."
    ),
    "generated_utc": utc_now_iso(),
    "split_fingerprint": stage1_fingerprint,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "max_features_per_set": MAX_FEATURES_PER_SET,
    "cohort": {
        "n_rows_common_cohort": reconstructed_rows,
        "n_classes_common": reconstructed_classes,
    },
    "sets": feature_set_definitions,
}
definitions_path = dump_json(output_dir / "feature_set_definitions.json", definitions_payload)

membership_path = output_dir / "feature_set_membership.csv"
stamp(feature_set_membership).to_csv(membership_path, index=False)

print(f"written: {summary_path.name} ({summary_path.stat().st_size} bytes)")
print(f"written: {definitions_path.name} ({definitions_path.stat().st_size} bytes)")
print(f"written: {membership_path.name} ({membership_path.stat().st_size} bytes)")
print()
print("All artefacts are aggregate and carry the validity stamp.")


written: correlation_summary.json (9883 bytes)
written: feature_set_definitions.json (23069 bytes)
written: feature_set_membership.csv (9471 bytes)

All artefacts are aggregate and carry the validity stamp.


In [23]:
# =============================================================================
#  STEP 7 — WRITE THE RECONSTRUCTION REPORT
#  The run report: what was read, what was checked, what was computed, and what
#  the results do and do not mean.
# =============================================================================
pairs_table_rows = []
for _, row in correlation_pairs.head(10).iterrows():
    pairs_table_rows.append(
        f"| `{row['feature_a']}` | `{row['feature_b']}` | {row['pearson']:+.4f} | "
        f"{row['spearman']:+.4f} | {'yes' if row['redundant_either'] else 'no'} |"
    )
pairs_table = "\n".join(pairs_table_rows) if pairs_table_rows else "| - | - | - | - | - |"

risk_features = [f for f in shortlist if is_absolute_statistic(f)]
risk_line = ", ".join(f"`{f}`" for f in risk_features) if risk_features else "(none)"

set_sections = []
for set_name, definition in feature_set_definitions.items():
    member_lines = "\n".join(
        f"| {m['position']} | `{m['feature']}` | {m['ordering_rank']:.1f} | "
        f"{m['max_abs_pearson_vs_set']:.3f} | {m['max_abs_spearman_vs_set']:.3f} | "
        f"{'yes' if m['potential_recording_or_session_risk'] else 'no'} |"
        for m in definition["members"]
    ) or "| - | - | - | - | - | - |"
    rejected_lines = "\n".join(
        f"| `{r['feature']}` | {r['ordering_rank']:.1f} | {r['rejection_reason']} |"
        for r in definition["rejected"]
    ) or "| - | - | - |"
    set_sections.append(
        f"""### `{set_name}`

* **Status:** `{definition['status']}` — evaluated: no, scored: no
* **Ordering rule:** {definition['ordering_rule']}
* **Tie-break:** {definition['tie_break']}
* **Acceptance rule:** {definition['acceptance_rule']}
* **Size:** {definition['n_features']} feature(s)

| # | Feature | Ordering rank | max \\|Pearson\\| vs set | max \\|Spearman\\| vs set | Position/time risk |
| ---: | --- | ---: | ---: | ---: | --- |
{member_lines}

| Rejected feature | Ordering rank | Reason |
| --- | ---: | --- |
{rejected_lines}"""
    )
sets_block = "\n\n".join(set_sections)

reconstruction_block = "\n".join(
    f"| `{entry['check']}` | {entry['stage1']!r} | {entry['reconstructed']!r} | "
    f"{'PASS' if entry['passed'] else 'FAIL'} |"
    for entry in reconstruction_checks
)

report = f"""# V3 Stage 3 — Feature Correlation and Candidate Feature-Set Definition

**Generated:** {utc_now_iso()}
**Validity:** {RESULTS_VALIDITY}
**Split fingerprint:** `{stage1_fingerprint}`

---

## 1. What this run did

1. Read and validated the stage-1 aggregates, the stage-2 aggregates and the raw course CSV
   files.
2. Rebuilt the stage-1 **complete-case cohort** from the raw data and cross-checked it against
   the stage-1 manifest.
3. Computed **Pearson and Spearman** correlations between the {len(shortlist)} features that
   stage 2 shortlisted, on that cohort.
4. Built **three candidate feature sets** with a deterministic, low-redundancy greedy rule.

**It did not train a model, did not score a feature set and did not select a champion.** The sets
are candidates for a later evaluation stage.

## 2. Inputs and validation

| Input | Files | Status |
| --- | ---: | --- |
| Stage 1 aggregates | {len(REQUIRED_STAGE1_FILES)} | {'all present' if all((stage1_dir / n).exists() for n in REQUIRED_STAGE1_FILES) else 'missing'} |
| Stage 2 aggregates | {len(REQUIRED_STAGE2_FILES)} | {'all present' if all((stage2_dir / n).exists() for n in REQUIRED_STAGE2_FILES) else 'missing'} |
| Raw course CSV files | {len(raw_files)} matched, {len(raw_files) - len(skipped_raw_files)} used | read in the runtime only |

Input-contract checks: **{len(validation_results) - len([e for e in validation_results if not e['passed']])} of
{len(validation_results)} passed**. The stage-1 and stage-2 split fingerprints agree, and every
shortlisted feature is both a stage-1 candidate column and a screened stage-1 feature.

Input digests (SHA-256) are recorded in `correlation_summary.json`. The raw file set is recorded
as a digest of the file names, so that the participant-linked names themselves stay out of the
artefacts.

## 3. Cohort reconstruction

The cohort is rebuilt from the raw data with the stage-1 rule: the label is taken from the CSV
content only, classes with fewer than {MIN_CLASS_COUNT} rows are dropped, and a row survives only
when **every** stage-1 candidate column is finite after numeric coercion (±inf counts as
missing).

| Check | Stage 1 | Rebuilt here | Result |
| --- | --- | --- | --- |
{reconstruction_block}

Rows merged: **{len(raw_frame)}** · cohort rows: **{reconstructed_rows}** · cohort classes:
**{reconstructed_classes}** · candidate columns: **{len(stage1_candidate_columns)}**

Because the rebuild matches the stage-1 manifest, the correlations below describe the same rows
the stage-1 metrics were computed on.

## 4. Correlation results

| Quantity | Value |
| --- | ---: |
| Features correlated | {len(shortlist)} |
| Pairs (upper triangle) | {n_pairs} |
| Pairs at or above the redundancy threshold ({CORRELATION_THRESHOLD}) | {n_redundant_pairs} |
| Maximum \\|Pearson\\| | {max_abs_pearson:.4f} |
| Maximum \\|Spearman\\| | {max_abs_spearman:.4f} |

The ten strongest pairs:

| Feature A | Feature B | Pearson | Spearman | Redundant |
| --- | --- | ---: | ---: | --- |
{pairs_table}

Full matrices: `pearson_correlation.csv`, `spearman_correlation.csv`. Every pair, with both
coefficients and the redundancy flags: `correlation_pairs.csv`.

### Features flagged `potential_recording_or_session_risk`

{risk_line}

These are absolute position or time statistics — the aggregate counterparts of the raw
coordinates and timestamps that stage 1 excluded. They may encode **where on the screen** and
**when in the recording** a segment occurred, that is, the recording rather than the person.
They are kept as provisional candidates and flagged wherever they appear, but they are **not**
described as robust identity-behaviour features, and their presence in a set is not evidence of
behavioural information.

## 5. The three candidate feature sets

{sets_block}

Every rejected feature is recorded in `feature_set_membership.csv` with its reason. No feature
was silently dropped and none was silently substituted. A set that could not be built from at
least two non-redundant candidates is written as `{SET_STATUS_INSUFFICIENT}` and is reported as
such.

## 6. What these results do and do not mean

* **Correlation is a constraint, not a signal.** A high correlation says two features carry
  overlapping information; it says nothing about whether either is useful for identification. The
  screening that ranked the candidates came from stage 1, and the stage-1 metrics remain the
  evidence of information.
* **This is not the stage-1 rank agreement.** The stage-2 figures compared how two models *order*
  the features. This stage measures how the feature *values* co-vary. The two quantities are
  unrelated and must not be conflated.
* **No set has been evaluated.** The sets differ in their ordering rule, therefore in their
  members; nothing here shows that one of them is better. That is the job of the next stage.
* **One cohort, one split.** The correlations inherit the complete-case bias and the split
  uncertainty of stage 1.

## 7. Outputs

| File | Content |
| --- | --- |
| `pearson_correlation.csv` | Pearson matrix over the shortlisted features |
| `spearman_correlation.csv` | Spearman matrix over the shortlisted features |
| `correlation_pairs.csv` | Upper triangle with both coefficients and the redundancy flags |
| `correlation_summary.json` | Configuration, input hashes, validation, counts, threshold, limitations |
| `feature_set_definitions.json` | The three sets, their members, their rejected features and the reasons |
| `feature_set_membership.csv` | One row per (set × feature), including the rejected ones |
| `correlation_reconstruction_report.md` | This report |
| `figures/pearson_heatmap.png`, `figures/spearman_heatmap.png`, `figures/redundant_pairs.png` | The figures |

The two matrix files carry a constant `results_validity` column so that the validity notice
travels with the file; drop it before using a matrix numerically.

## 8. Next steps

1. **Supervisor review** of this report, in particular of the `potential_recording_or_session_risk`
   flags and of whether the three sets are the right candidates at all.
2. **A restricted-pool comparison**: how much of the stage-1 signal survives when the absolute
   position and time statistics are removed from the candidate pool.
3. **Stage 4 — feature-set evaluation**: the three sets, the same cohort and the same protocol,
   with the dummy floor **and** the chance reference reported, and the risk flags carried into the
   discussion.
4. **A session-aware evaluation** if any recording identifier becomes available.

## 9. Statement of validity

Every number in this report comes from the run identified above and is **only valid after a real
Colab run on the real course data**. The notebook is committed without stored outputs and without
any result value. No raw data, per-row value, label value, source file name or absolute path
appears in any artefact produced by this stage.
"""

report_path = output_dir / "correlation_reconstruction_report.md"
report_path.write_text(report, encoding="utf-8")

print(f"written: {report_path.name} ({report_path.stat().st_size} bytes)")
print()
print("Output directory:", output_dir)
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(output_dir)}")


written: correlation_reconstruction_report.md (11414 bytes)

Output directory: /content/drive/MyDrive/CP5046_Mouse_Dynamics/V3/stage3_output
  correlation_pairs.csv
  correlation_reconstruction_report.md
  correlation_summary.json
  feature_set_definitions.json
  feature_set_membership.csv
  figures/pearson_heatmap.png
  figures/redundant_pairs.png
  figures/spearman_heatmap.png
  pearson_correlation.csv
  spearman_correlation.csv


## Output reference

Everything is written to `OUTPUT_DIR`. All artefacts are aggregate, carry the validity stamp
`only valid after a real Colab run`, and contain **no raw row, no per-row value, no label value,
no source file name and no absolute path**.

| File | Content |
| --- | --- |
| `pearson_correlation.csv` | Pearson matrix over the shortlisted features, indexed by `feature` |
| `spearman_correlation.csv` | Spearman matrix over the shortlisted features, indexed by `feature` |
| `correlation_pairs.csv` | Upper triangle: `feature_a`, `feature_b`, `pearson`, `spearman`, `abs_pearson`, `abs_spearman`, `redundant_pearson`, `redundant_spearman`, `redundant_either`, `correlation_defined`, and the risk flags |
| `correlation_summary.json` | Configuration, input hashes, the validation report, the reconstruction cross-check, counts, the threshold and the limitations |
| `feature_set_definitions.json` | The three sets with their members, their rejected features, the ordering rule, the tie-break and the acceptance rule |
| `feature_set_membership.csv` | One row per (set × feature), including the rejected ones and the reason |
| `correlation_reconstruction_report.md` | The run report |
| `figures/pearson_heatmap.png`, `figures/spearman_heatmap.png`, `figures/redundant_pairs.png` | The figures |

**Reading the matrix files.** They carry a constant `results_validity` column so that the validity
notice travels with the file:

```python
pearson = pd.read_csv("pearson_correlation.csv", index_col=0)
pearson = pearson.drop(columns=["results_validity"])   # before numeric use
```

---

## Limitations of this stage

1. **Nothing here is a result about performance.** No model was trained, no set was scored and no
   champion was chosen. The sets are **provisional / not yet evaluated** candidates.
2. **Correlation is a redundancy constraint, not evidence of information.** Two features can be
   uncorrelated and both useless, or highly correlated and both useful. The evidence of
   information remains the stage-1 held-out metrics.
3. **The correlations are not the stage-1 rank agreement.** The stage-2 figures compared how two
   models *ordered* the features; this stage measures how the feature *values* co-vary. Confusing
   the two would misrepresent both.
4. **One cohort, one split.** The complete-case cohort is a subsample of the merged data, and the
   rows it drops are not missing at random. No bootstrap interval is reported, so a coefficient
   close to the threshold should be read with that uncertainty in mind.
5. **The risk flags are a warning, not a verdict.** `potential_recording_or_session_risk` marks
   absolute position and time statistics. They are kept because the decision belongs to the
   supervisor, and they are flagged everywhere they appear — but their presence in a set is not
   evidence of behavioural information.
6. **The greedy construction is order-dependent by design.** A different ordering rule would give
   a different set, and that is exactly what the three sets explore. The acceptance rule and the
   tie-break are identical across the three, so the sets are comparable.
7. **The threshold is a single number.** Pearson and Spearman are compared against the same
   threshold, and the two coefficients can disagree; both are reported for every pair so that a
   disagreement is visible rather than hidden.

## What the supervisor must confirm

- [ ] The input-contract report shows every check as PASS, and the reconstruction cross-check
      matches the stage-1 manifest (rows, classes and the protocol constants).
- [ ] The stage-1 and stage-2 split fingerprints agree, so the correlations describe the same
      rows as the stage-1 metrics.
- [ ] The correlation threshold of 0.85 and `MAX_FEATURES_PER_SET` are the values the project
      wants before the sets are evaluated.
- [ ] The three sets are the right candidates, and the features rejected for redundancy are
      rejected for a good reason.
- [ ] The `potential_recording_or_session_risk` features are handled deliberately — kept as
      provisional, or excluded from the evaluation entirely.
- [ ] No set is described as the champion before stage 4 evaluates them.
- [ ] Nothing raw, participant-linked or path-bearing has entered the outputs.

**Stage 3 ends here.** The feature-set evaluation notebook is a separate deliverable and is not
created by this file.
